# 곱의 미분법과 연쇄법칙

> 미적분 5강 · 곱의 미분법과 연쇄법칙

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [곱의 미분법과 연쇄법칙](https://mioon1402.github.io/timeseriesdata/calc/C05-product-chain.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 왜 그냥 곱하면 안 되나

## 1. 곱의 미분법 — 팽창하는 직사각형의 해부

## 2. 따름정리 — 상수배와 합

## 3. 연쇄법칙 — 변화의 도미노

## 4. 기어비로 보는 연쇄법칙

## 5. 실전: 두 규칙을 함께 쓰기

## 6. 몫의 미분법 (보너스)

## 7. 파이썬으로 확인하기

**5-1. 세 조각으로 쪼개보기**

In [ ]:
import numpy as np

g  = lambda x: x**2 + 1
h  = lambda x: np.sin(x) + 2
gp = lambda x: 2*x
hp = lambda x: np.cos(x)

x = 1.0
print(f"g({x}) = {g(x):.6f}   h({x}) = {h(x):.6f}\n")
print(f"{'dx':>7} {'오른쪽 띠 h·dg':>16} {'위쪽 띠 g·dh':>16} {'모서리 dg·dh':>15} {'변화율':>12}")
for dx in [0.5, 0.1, 0.01, 0.001]:
    dg = g(x+dx) - g(x)
    dh = h(x+dx) - h(x)
    전체 = g(x+dx)*h(x+dx) - g(x)*h(x)
    assert abs(h(x)*dg + g(x)*dh + dg*dh - 전체) < 1e-12    # 세 조각의 합이 정확히 전체
    print(f"{dx:>7} {h(x)*dg:>16.6f} {g(x)*dh:>16.6f} {dg*dh:>15.6f} {전체/dx:>12.6f}")

print(f"\ng'h + gh' = {gp(x)*h(x) + g(x)*hp(x):.6f}   ← 변화율이 여기로 다가간다")

**5-2. (gh)′ = g′h′ 은 왜 틀리나**

In [ ]:
def 수치미분(f, x, h=1e-6):
    return (f(x+h) - f(x-h)) / (2*h)

x = 3.0
f = lambda t: t * t                       # x · x

print(f"진짜 (x·x)′ at x={x} :", 수치미분(f, x))
print(f"틀린 계산 (x)′·(x)′  :", 1 * 1)
print(f"곱의 법칙 g′h + gh′  :", 1*x + x*1)
print("\n→ 미분은 곱셈을 그냥 통과하지 않는다. 두 기여를 '더해야' 한다.")

**5-3. 연쇄법칙 — 기어비의 곱**

In [ ]:
import numpy as np

사슬 = {
    "sin(x³)":  (lambda x: x**3,      lambda x: 3*x**2,
                 np.sin,              np.cos),
    "(sin x)²": (np.sin,               np.cos,
                 lambda u: u**2,       lambda u: 2*u),
    "e^(x²)":   (lambda x: x**2,       lambda x: 2*x,
                 np.exp,               np.exp),
}

x = 1.1
print(f"x = {x} 에서\n")
print(f"{'사슬':>12} {'안 g′':>10} {'바깥 h′(g)':>12} {'곱':>12} {'수치미분':>14}")
for 이름, (g, gp, h, hp) in 사슬.items():
    전체 = lambda t, g=g, h=h: h(g(t))
    수치 = (전체(x+1e-6) - 전체(x-1e-6)) / 2e-6
    print(f"{이름:>12} {gp(x):>10.5f} {hp(g(x)):>12.5f} {hp(g(x))*gp(x):>12.5f} {수치:>14.5f}")

print("\n→ 두 기어비의 곱이 전체 기어비다.")

**5-4. 곱 + 연쇄 조립**

In [ ]:
import numpy as np

문제들 = [
    ("x²·sin(x³)",
     lambda x: x**2 * np.sin(x**3),
     lambda x: 2*x*np.sin(x**3) + 3*x**4*np.cos(x**3)),
    ("x·cos(x²)",
     lambda x: x*np.cos(x**2),
     lambda x: np.cos(x**2) - 2*x**2*np.sin(x**2)),
    ("(x²+1)(sin x + 2)",
     lambda x: (x**2+1)*(np.sin(x)+2),
     lambda x: 2*x*(np.sin(x)+2) + (x**2+1)*np.cos(x)),
    ("sin(3x)",
     lambda x: np.sin(3*x),
     lambda x: 3*np.cos(3*x)),
]

print(f"{'식':>22} {'x':>6} {'손으로':>14} {'수치미분':>14} {'차':>10}")
for 이름, f, fp in 문제들:
    for x in [0.7, 1.4]:
        수치 = (f(x+1e-6) - f(x-1e-6)) / 2e-6
        print(f"{이름:>22} {x:>6} {fp(x):>14.8f} {수치:>14.8f} {fp(x)-수치:>10.1e}")

**5-5. sympy 로 검산하기**

In [ ]:
import sympy as sp

x = sp.Symbol('x')
식들 = [
    x**2 * sp.sin(x**3),
    x * sp.cos(x**2),
    (x**2 + 1) * (sp.sin(x) + 2),
    sp.sin(x) / (x**2 + 1),          # 몫
]

for 식 in 식들:
    print(f"{str(식):>26}  →  {sp.simplify(sp.diff(식, x))}")

print("\n→ 손으로 조립한 결과와 같은지 확인해보세요.")

**5-6. 연습문제**

In [ ]:
# 문제 1. (x³ + 2x)(cos x) 를 미분하세요. 5-4 셀의 방식으로 확인해보세요.

# 문제 2. sin(sin(x)) 를 미분하세요. (연쇄법칙 두 번? 아니면 한 번?)

# 문제 3. e^(sin(x²)) 를 미분하세요. 사슬이 세 겹입니다.

# 문제 4. (gh)′ = g′h′ 이 성립하는 g, h 가 있을까요? 있다면 어떤 경우일까요?

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
import numpy as np
수치 = lambda f, x: (f(x+1e-6) - f(x-1e-6)) / 2e-6

# 문제 1 — 곱의 법칙
f1  = lambda x: (x**3 + 2*x) * np.cos(x)
f1p = lambda x: (3*x**2 + 2)*np.cos(x) - (x**3 + 2*x)*np.sin(x)
print(f"문제 1: 손 {f1p(1.2):.8f}   수치 {수치(f1, 1.2):.8f}\n")

# 문제 2 — 연쇄법칙 한 번 (바깥 sin, 안 sin)
f2  = lambda x: np.sin(np.sin(x))
f2p = lambda x: np.cos(np.sin(x)) * np.cos(x)
print(f"문제 2: 손 {f2p(1.2):.8f}   수치 {수치(f2, 1.2):.8f}")
print("        cos(sin x)·cos x — 바깥 미분(안은 그대로) × 안 미분\n")

# 문제 3 — 사슬 세 겹: e^u, u = sin v, v = x²
f3  = lambda x: np.exp(np.sin(x**2))
f3p = lambda x: np.exp(np.sin(x**2)) * np.cos(x**2) * 2*x
print(f"문제 3: 손 {f3p(1.2):.8f}   수치 {수치(f3, 1.2):.8f}")
print("        기어비를 세 개 곱하면 된다\n")

# 문제 4 — 지수함수 꼴에서 우연히 성립할 수 있다
print("문제 4: g′h + gh′ = g′h′ 이 되려면 특별한 관계가 필요하다.")
print("        예: g = h = e^(2x) 이면 좌변 = 2e²ˣ·e²ˣ·2 = 4e⁴ˣ,")
print("        우변 = 2e²ˣ·2e²ˣ = 4e⁴ˣ — 같다!")
g4 = h4 = lambda x: np.exp(2*x)
print(f"        확인 x=0.5: 곱의법칙 {수치(lambda t: g4(t)*h4(t), 0.5):.6f}"
      f"   g′h′ {수치(g4,0.5)*수치(h4,0.5):.6f}")
print("        → '항상 틀린' 게 아니라 '일반적으로 틀린' 것이다. 우연을 규칙으로 착각하면 안 된다.")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)